In [1]:
import pandas as pd
import re
import emoji
import emot

emot_obj = emot.core.emot()
INPUT_FILE = 'cleaned_ott.csv' 
OUTPUT_FILE = 'cleaned_data_for_training.csv'

def get_first_feature_tag(meaning):
    first_part = meaning.split(',')[0]
    
    first_part = first_part.split(' or ')[0]
    
    clean_tag = first_part.strip().replace(' ', '_')
    
    return f" :{clean_tag}: "
    
def convert_emoticons(text):
    res = emot_obj.emoticons(text)
    
    if not res['flag']:
        return text
    
    for i in range(len(res['value'])):
        emoticon = res['value'][i]
        meaning = res['mean'][i]
        
        clean_meaning = get_first_feature_tag(meaning)
        escaped_emoticon = re.escape(emoticon)
        text = re.sub(escaped_emoticon, clean_meaning, text)
        
    return text

def clean_text(text):
    
    # 1
    if text is None:
        return ""
    text = str(text)
    
    # 2
    text = emoji.demojize(text, delimiters=(" :", ": ")) 
    
    # 3
    text = convert_emoticons(text)
    
    # 4
    text = re.sub(r'<.*?>', '', text)
    
    # 5
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

In [17]:
def main():
    print(f"Loading data from '{INPUT_FILE}'...")
    try:
        df = pd.read_csv(INPUT_FILE)
    except FileNotFoundError:
        print(f"ERROR: Could not find {INPUT_FILE}. Did you run Step 1?")
        return

    print("Processing reviews")
    df['clean_text'] = df['review_text'].apply(clean_text)

    print(f"Saving cleaned data to '{OUTPUT_FILE}'...")
    df.to_csv(OUTPUT_FILE, index=False)
    
    print("\n✅ PREPROCESSING COMPLETE.")
    print(f"Original: {df['review_text'].iloc[0][:50]}...")
    print(f"Cleaned:  {df['clean_text'].iloc[0][:50]}...")


if __name__ == "__main__":
    main()

Loading data from 'cleaned_ott.csv'...
Processing reviews
Saving cleaned data to 'cleaned_data_for_training.csv'...

✅ PREPROCESSING COMPLETE.
Original: We stayed at the Schicago Hilton for 4 days and 3 ...
Cleaned:  We stayed at the Schicago Hilton for 4 days and 3 ...
